In [0]:
from pyspark.sql.functions import explode, sequence, to_date, col, year, month, dayofweek, weekofyear, date_format

# Intervalo de datas
inicio = "2024-01-01"
fim = "2025-12-31"

# Geração de Sequencia de Datas
df_data = spark.sql(f"SELECT sequence(to_date('{inicio}'), to_date('{fim}'), interval 1 day) as intervalo") \
    .withColumn("data", explode(col("intervalo")))

dim_data = df_data.select(
    col("data").alias("id_data"),
    year(col("data")).alias("ano"),
    month(col("data")).alias("mes"),
    date_format(col("data"), "MMMM").alias("nome_mes"),
    weekofyear(col("data")).alias("semana_ano"),
    dayofweek(col("data")).alias("dia_semana"),
    date_format(col("data"), "EEEE").alias("nome_dia_semana")
)

# Salvando tabela Gold
dim_data.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("gold_dim_data")

display(dim_data.limit(5))